# 17. Visual Representations

픽셀 이동, patch 표현, 정규화 거리와 contrastive matching을 직접 계산합니다.

핵심 실습은 NumPy와 Python 표준 라이브러리만 사용합니다. API 호출, 모델 다운로드, GPU는 사용하지 않습니다. 수치는 고정된 작은 입력의 계산이며 실제 pretrained model의 성능이 아닙니다.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(27)

## 픽셀에서 patch까지

동일한 모양을 한 칸 옮겼을 때 픽셀 거리가 어떻게 달라지는지 봅니다.

In [2]:
image = np.zeros((4, 4))
image[1:3, 1:3] = 1
shifted = np.roll(image, 1, axis=1)
print("squared pixel distance:", np.square(image-shifted).sum())
patches = image.reshape(2, 2, 2, 2).transpose(0, 2, 1, 3).reshape(4, 4)
print("patches shape:", patches.shape)
print(patches)
assert np.allclose(patches.sum(), image.sum())

squared pixel distance: 4.0
patches shape: (4, 4)
[[0. 0. 0. 1.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]]


## 정규화된 embedding 검색

아래 벡터는 학습된 visual embedding의 비교 연산을 분리한 입력입니다. 실제 이미지를 의미로 encoding하는 모델은 아닙니다.

In [3]:
def normalize(z):
    n = np.linalg.norm(z, axis=-1, keepdims=True)
    return z / np.maximum(n, 1e-12)
query = normalize(np.array([[1., 0.]]))
gallery = normalize(np.array([[.8,.6], [0.,1.], [-1.,0.]]))
cosine = query @ gallery.T
distance2 = np.square(query[:,None,:] - gallery[None,:,:]).sum(axis=-1)
print("cosine:", cosine)
print("squared distance:", distance2)
assert np.array_equal(np.argsort(-cosine), np.argsort(distance2))

cosine: [[ 0.8  0.  -1. ]]
squared distance: [[0.4 2.  4. ]]


## Contrastive loss와 collapse

짝이 맞는 두 view의 similarity matrix를 만듭니다. 모든 벡터가 같아지면 retrieval이 입력을 구별할 수 없습니다.

In [4]:
def contrastive_loss(a, b, temperature=.2):
    logits = normalize(a) @ normalize(b).T / temperature
    logits -= logits.max(axis=1, keepdims=True)
    probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
    return -np.log(np.diag(probs)).mean(), probs
a = np.array([[1.,0.], [0.,1.], [-1.,0.]])
b = np.array([[.9,.1], [.1,.9], [-.9,.1]])
loss, probs = contrastive_loss(a,b)
collapsed_loss, collapsed_probs = contrastive_loss(np.ones((3,2)), np.ones((3,2)))
print("matched loss:", loss, "collapsed loss:", collapsed_loss)
print("collapsed probabilities:\n", collapsed_probs)
assert loss < collapsed_loss
assert np.allclose(collapsed_probs, 1/3)

matched loss: 0.013310057339733098 collapsed loss: 1.0986122886681098
collapsed probabilities:
 [[0.3333 0.3333 0.3333]
 [0.3333 0.3333 0.3333]
 [0.3333 0.3333 0.3333]]


## 과제

- gallery 벡터의 크기를 10배로 바꿔 cosine 순위가 유지되는지 확인하세요. 정규화하지 않은 내적과 비교하세요.
- 색깔로 라벨이 결정되는 데이터에서 색 정보를 지우는 augmentation이 어떤 정보를 없애는지 작은 배열로 보이세요.

In [5]:
# 과제 풀이 1
scaled = gallery * np.array([[10.], [1.], [1.]])
assert np.allclose(query @ normalize(scaled).T, cosine)
print("raw dot product:", query @ scaled.T)
# 과제 풀이 2: 같은 밝기의 빨강/초록 신호
colors = np.array([[1.,0.], [0.,1.]])
gray = colors.mean(axis=1)
assert gray[0] == gray[1]
print("distinct colors, identical gray:", gray)

raw dot product: [[ 8.  0. -1.]]
distinct colors, identical gray: [0.5 0.5]


## 해석

cosine 계산이 맞다고 표현이 과제에 적합하다는 뜻은 아닙니다. 실제 encoder를 비교할 때는 같은 split, preprocessing, readout을 고정하고 nearest example을 직접 확인합니다. 참고: [ViT](https://arxiv.org/abs/2010.11929), [DINO](https://arxiv.org/abs/2104.14294).